In [7]:
import cv2
import mediapipe as mp
import torch
import numpy as np

print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)
print("PyTorch:", torch.__version__)

OpenCV: 4.11.0
MediaPipe: 0.10.21
PyTorch: 2.15.0+cpu


In [ ]:
import torch
import time

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.15.0+cpu
CUDA available: False


In [ ]:
%pip install torch torchvision

In [ ]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Cannot access webcam")


mpHands = mp.solutions.hands
mpDraw = mp.solutions.drawing_utils

hands = mpHands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    model_complexity=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

pTime = 0

try:
    while True:

        success, img = cap.read()

        if not success:
            print("Failed to capture frame")
            break

        
        imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        
        results = hands.process(imgRGB)

        
        all_landmarks = []

        if results.multi_hand_landmarks:

            for hand_index, handLms in enumerate(
                results.multi_hand_landmarks
            ):

                hand_data = []

                for lm in handLms.landmark:

                    h, w, c = img.shape

                    cx = int(lm.x * w)
                    cy = int(lm.y * h)

                    
                    hand_data.append([lm.x, lm.y, lm.z])

                all_landmarks.append(hand_data)

                
                mpDraw.draw_landmarks(
                    img,
                    handLms,
                    mpHands.HAND_CONNECTIONS
                )

                
                wrist = handLms.landmark[0]

                cx = int(wrist.x * w)
                cy = int(wrist.y * h)

                cv2.circle(
                    img,
                    (cx, cy),
                    10,
                    (255, 0, 255),
                    cv2.FILLED
                )

        
        cTime = time.time()

        fps = 1 / (cTime - pTime) if pTime != 0 else 0
        pTime = cTime

        cv2.putText(
            img,
            f"FPS: {int(fps)}",
            (10, 70),
            cv2.FONT_HERSHEY_COMPLEX,
            1,
            (255, 0, 255),
            2
        )

        cv2.imshow("Sign Language - Hand Tracking", img)

    
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

finally:
    cap.release()
    hands.close()
    cv2.destroyAllWindows()
